<style>.md-sidebar--secondary {display: none !important;}</style>

# Basic Simulation

In this example, we will demonstrate some of the basics of PassengerSim.
We'll interface with the tool through the `passengersim` Python interface,
which is imported into Python in the same manner as any other Python library.

In [ ]:
import passengersim as pax

pax.versions()

Using PassengerSim generally involves three steps: configuring a simulation, 
running it, and analyzing the results.  The first and last of these steps 
rely primarily on the open-source `passengersim` code that runs entirely in
Python, while running a simulation relies on the `passengersim.core` library, 
which is the proprietary licensed code.

To get started, we will load a PassengerSim configuration from YAML input 
files. These files are structured plain text inputs that define everything
needed to run a simulation: carriers, flight legs, passenger choice models, 
simulation controls, and more.  The demo YAML file shown below shows how 
one of these YAML input files might be structured, with "include" statements
that point to files containing much of the fundamental network configuration
and a handful of specific settings can supplement or replace settings from the
included files.

In [ ]:
from passengersim.utils.codeview import show_file

show_file(pax.demo_network("3MKT/DEMO"))

We can load the YAML files into a `Config` object, which will both handle loading
the raw content and validating that it includes everything necessary to prepare
a PassengerSim simulation.

In [ ]:
cfg = pax.Config.from_yaml(pax.demo_network("3MKT/DEMO"))

After loading, it's possible to modify the `Config` object in Python to change aspects
of a simulation. For this example, we won't change anything here, and just create the 
`Simulation` object from the existing `Config`.  We can choose to create a regular 
`Simulation` which will run all trials and samples sequentially in a single process, or 
a `MultiSimulation` which will run each trial in parallel, with each trial running in 
a seperate subprocess.  The later option is generally much faster on multi-core computers,
but detailed simulation model development or debugging can usually be done more conveniently
in the single process workflow.

Since our demo is configured to run two trials, we'll run a `MultiSimulation`.

In [ ]:
sim = pax.MultiSimulation(cfg)

Running the simulation is as simple as calling the `run` command, which runs the simulation and returns a summary output object.

In [ ]:
summary = sim.run()

By default, PassengerSim will generate a summary output that includes a wide variety of 
aggregated summary output data, which can be reviewed or visualized using an included 
suite of analysis tools.

In [ ]:
summary

The raw data of the summary tables shown can be accessed via direct attribute access
on the `SimulationTables` object.  For example, we can see the summary data collected 
on the legs like this:

In [ ]:
summary.legs

In addition to accessing the raw data, PassengerSim also has a built-in collection of 
visualization tools for reviewing the data.  These tools include aggregate summary
figures, e.g. some figures showing carrier revenue, load factors, or RASM.

In [ ]:
summary.fig_carrier_revenues()

In [ ]:
summary.fig_carrier_load_factors()

In [ ]:
summary.fig_carrier_rasm()

We can also examine more detailed slices of simulation results.  One commonly 
used visualization show the fare class mix across all bookings on each carrier.

In [ ]:
summary.fig_fare_class_mix()

We can also dive deeper into this fare class data, to look not just at 
how many bookings are in each fare class, but also when those bookings 
occured and what passenger segments they came from.  The figure below
clearly shows that the lower fare classes are booked almost exclusively 
by leisure travlers, and primarily in the early time frames.

In [ ]:
summary.fig_bookings_by_timeframe(by_class=True, by_carrier="AL1")

We can also do a deeper dive into the passengers on an individual leg, to look
at the breakdown of their path origins and destinations, and the fare classes
sold on that particular leg.

In [ ]:
summary.fig_select_leg_analysis(101)

A different perspective on the results can be seen in this figure of carrier 
revenue results, which are plotted sample-by-sample. Here we can see how 
carrier do against each other: if AL1 is having a good day, is AL2 also having
a similarly good day?

In [ ]:
summary.fig_carrier_head_to_head_revenue("AL1", "AL2")